# VideoGen Server — Google Colab
Runs LTX-Video on a free T4 GPU. Uses Cloudflare Tunnel — **no sign-up needed**.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells top to bottom (Cell 1 → Cell 6)
3. Cell 6 prints a `trycloudflare.com` URL — copy it into `VideoGenApiClient.kt`
4. Rebuild and reinstall the APK

> **If session crashes** — just re-run Cell 5 and Cell 6. Model stays loaded.


In [ ]:
# ── Cell 1: Check GPU & RAM ────────────────────────────────────────────────
!nvidia-smi
import psutil
ram = psutil.virtual_memory()
print(f"RAM: {ram.available/1e9:.1f} GB free of {ram.total/1e9:.1f} GB")

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────
!pip install -q \
    fastapi "uvicorn[standard]" \
    "diffusers>=0.33.0" transformers accelerate sentencepiece \
    torch torchvision torchaudio \
    imageio imageio-ffmpeg python-multipart \
    openai-whisper librosa soundfile
print("Dependencies installed.")

In [ ]:
# ── Cell 3: Install Cloudflare Tunnel (no account needed) ──────────────────
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# ── Cell 4: Load LTX-Video model (memory-optimised for free T4) ────────────
import torch
from diffusers import LTXPipeline

# Free up any leftover GPU memory before loading
torch.cuda.empty_cache()

MODEL_ID = "Lightricks/LTX-Video-0.9.1"
print(f"Loading {MODEL_ID} ...  (first run downloads ~5 GB, ~5 min)")

pipe = LTXPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)
pipe.enable_model_cpu_offload()
pipe.vae.enable_slicing()   # decode one frame at a time → saves ~2 GB RAM
pipe.vae.enable_tiling()    # tile large frames → saves more RAM

import gc
gc.collect()
torch.cuda.empty_cache()

import psutil
ram = psutil.virtual_memory()
print(f"Model ready.  RAM free: {ram.available/1e9:.1f} GB")
print("IMPORTANT: if Colab crashes, re-run this cell first before Cell 5+6.")

In [ ]:
# ── Cell 5: FastAPI server definition ──────────────────────────────────────
import os, uuid, shutil, tempfile, threading, time, re, subprocess, gc
import torch  # re-import here so this cell is self-contained after a crash
from pathlib import Path
from typing import Optional, Dict, Any
from datetime import datetime

import numpy as np
import imageio_ffmpeg
from fastapi import FastAPI, HTTPException, BackgroundTasks, Header, Depends, File, Form, UploadFile
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

app = FastAPI(title="VideoGen Colab Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

jobs: Dict[str, Dict[str, Any]] = {}
API_KEY    = os.getenv("VIDEOGEN_API_KEY", "dev-secret-key")
OUTPUT_DIR = Path("/content/videogen_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def verify_api_key(x_api_key: str = Header(...)):
    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")
    return x_api_key


class GenerateRequest(BaseModel):
    prompt:              str             = Field(..., min_length=3, max_length=1000)
    negative_prompt:     Optional[str]   = "blurry, distorted, watermark, low quality"
    duration_seconds:    Optional[float] = Field(default=2.0, ge=1.0, le=8.0)
    fps:                 Optional[int]   = Field(default=8,   ge=8,   le=16)
    height:              Optional[int]   = Field(default=320)
    width:               Optional[int]   = Field(default=512)
    num_inference_steps: Optional[int]   = Field(default=25,  ge=10,  le=40)
    guidance_scale:      Optional[float] = Field(default=3.5, ge=1.0, le=10.0)
    seed:                Optional[int]   = None
    enhance_prompt:      bool            = True


class JobStatus(BaseModel):
    job_id:           str
    status:           str
    created_at:       str
    updated_at:       str
    progress:         int            = 0
    video_url:        Optional[str]  = None
    error:            Optional[str]  = None
    metadata:         Optional[dict] = None
    transcribed_text: Optional[str]  = None


def _frames_to_mp4(frames, out_path: str, fps: int, width: int, height: int):
    height = height - (height % 2)
    width  = width  - (width  % 2)
    writer = imageio_ffmpeg.write_frames(
        out_path, size=(width, height), fps=fps,
        codec="libx264", pix_fmt_in="rgb24", pix_fmt_out="yuv420p",
        ffmpeg_log_level="quiet",
        output_params=["-preset", "fast", "-crf", "23", "-movflags", "+faststart"],
    )
    writer.send(None)
    for frame in frames:
        arr = np.array(frame.resize((width, height))).astype(np.uint8)
        if arr.ndim == 2:
            arr = np.stack([arr] * 3, axis=-1)
        writer.send(arr.tobytes())
    writer.close()


def _run_generation(job_id: str, req: GenerateRequest):
    jobs[job_id]["status"]     = "processing"
    jobs[job_id]["updated_at"] = datetime.utcnow().isoformat()
    try:
        # Guard: catch the case where Cell 4 was never run / session restarted
        if "pipe" not in globals() or pipe is None:
            raise RuntimeError("Model not loaded. Re-run Cell 4 first.")

        n_frames = max(9, int(req.duration_seconds * req.fps))
        n_frames = ((n_frames - 1) // 8) * 8 + 1  # LTX-Video requires 8k+1 frames
        print(f"[Gen] '{req.prompt[:50]}'  frames={n_frames}  {req.width}x{req.height}  steps={req.num_inference_steps}")

        generator = torch.Generator(device="cpu").manual_seed(req.seed) if req.seed is not None else None
        result = pipe(
            prompt=req.prompt,
            negative_prompt=req.negative_prompt,
            num_frames=n_frames,
            height=req.height - (req.height % 32),
            width=req.width   - (req.width  % 32),
            num_inference_steps=req.num_inference_steps,
            guidance_scale=req.guidance_scale,
            generator=generator,
        )

        out_path = str(OUTPUT_DIR / f"{job_id}.mp4")
        _frames_to_mp4(result.frames[0], out_path, req.fps, req.width, req.height)

        del result
        gc.collect()
        torch.cuda.empty_cache()

        jobs[job_id].update({
            "status": "completed", "progress": 100,
            "video_path": out_path, "video_url": f"/videos/{job_id}",
            "updated_at": datetime.utcnow().isoformat(),
            "metadata": {"prompt": req.prompt, "model": "ltx-video", "frames": n_frames},
        })
        print(f"[Gen] Done → {out_path}")

    except Exception as e:
        gc.collect()
        torch.cuda.empty_cache()
        jobs[job_id].update({"status": "failed", "error": str(e),
                              "updated_at": datetime.utcnow().isoformat()})
        print(f"[Gen] FAILED: {e}")


@app.get("/health")
async def health():
    import psutil
    ram = psutil.virtual_memory()
    model_ok = "pipe" in globals() and pipe is not None
    return {"status": "ok", "model": "ltx-video", "model_loaded": model_ok,
            "ram_free_gb": round(ram.available / 1e9, 1),
            "active_jobs": sum(1 for j in jobs.values() if j["status"] == "processing"),
            "total_jobs": len(jobs)}

@app.post("/generate", response_model=JobStatus)
async def generate_video(req: GenerateRequest, background_tasks: BackgroundTasks,
                          _: str = Depends(verify_api_key)):
    if "pipe" not in globals() or pipe is None:
        raise HTTPException(status_code=503, detail="Model not loaded. Re-run Cell 4 in Colab.")
    if any(j["status"] == "processing" for j in jobs.values()):
        raise HTTPException(status_code=429, detail="A job is already processing.")
    job_id = str(uuid.uuid4())
    now = datetime.utcnow().isoformat()
    jobs[job_id] = {"job_id": job_id, "status": "queued",
                    "created_at": now, "updated_at": now, "progress": 0, "request": req.dict()}
    background_tasks.add_task(_run_generation, job_id, req)
    return JobStatus(**jobs[job_id])

@app.get("/jobs/{job_id}", response_model=JobStatus)
async def get_job(job_id: str, _: str = Depends(verify_api_key)):
    if job_id not in jobs:
        raise HTTPException(status_code=404, detail="Job not found")
    return JobStatus(**jobs[job_id])

@app.get("/videos/{job_id}")
async def download_video(job_id: str):
    if job_id not in jobs:
        raise HTTPException(status_code=404, detail="Job not found")
    job = jobs[job_id]
    if job["status"] != "completed":
        raise HTTPException(status_code=202, detail=f"Job status: {job['status']}")
    video_path = job.get("video_path")
    if not video_path or not Path(video_path).exists():
        raise HTTPException(status_code=404, detail="Video file not found")
    return FileResponse(video_path, media_type="video/mp4",
                        filename=f"videogen_{job_id}.mp4")

@app.get("/jobs")
async def list_jobs(limit: int = 20, _: str = Depends(verify_api_key)):
    sorted_jobs = sorted(jobs.values(), key=lambda j: j["created_at"], reverse=True)
    return {"jobs": sorted_jobs[:limit], "total": len(jobs)}

@app.delete("/jobs/{job_id}")
async def delete_job(job_id: str, _: str = Depends(verify_api_key)):
    if job_id not in jobs:
        raise HTTPException(status_code=404, detail="Job not found")
    job = jobs.pop(job_id)
    vp = job.get("video_path")
    if vp and Path(vp).exists():
        Path(vp).unlink()
    return {"deleted": job_id}

@app.post("/generate-from-audio", response_model=JobStatus)
async def generate_from_audio(
    background_tasks: BackgroundTasks,
    speech_file: UploadFile = File(...),
    music_file:  Optional[UploadFile] = File(default=None),
    duration_seconds: float = Form(default=2.0),
    language: str           = Form(default="auto"),
    seed: Optional[int]     = Form(default=None),
    _: str = Depends(verify_api_key),
):
    import whisper
    LANG_CODES = {"auto": None, "odia": "or", "telugu": "te", "hindi": "hi", "english": "en"}
    whisper_lang = LANG_CODES.get(language.lower())
    tmp_dir = tempfile.mkdtemp()
    try:
        s_suffix = Path(speech_file.filename or "audio.m4a").suffix or ".m4a"
        speech_path = os.path.join(tmp_dir, f"speech{s_suffix}")
        with open(speech_path, "wb") as f:
            f.write(await speech_file.read())
        if music_file:
            await music_file.read()
        wmodel = whisper.load_model("small")
        result = wmodel.transcribe(speech_path, language=whisper_lang)
        transcript = result.get("text", "").strip()
        del wmodel; gc.collect(); torch.cuda.empty_cache()
        if not transcript:
            raise HTTPException(status_code=422, detail="Could not transcribe audio.")
    except HTTPException:
        shutil.rmtree(tmp_dir, ignore_errors=True)
        raise
    except Exception as e:
        shutil.rmtree(tmp_dir, ignore_errors=True)
        raise HTTPException(status_code=500, detail=f"Audio error: {e}")
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)
    req = GenerateRequest(prompt=transcript,
                          duration_seconds=min(max(duration_seconds, 1.0), 8.0), seed=seed)
    job_id = str(uuid.uuid4())
    now = datetime.utcnow().isoformat()
    jobs[job_id] = {"job_id": job_id, "status": "queued",
                    "created_at": now, "updated_at": now,
                    "progress": 0, "request": req.dict(), "transcribed_text": transcript}
    background_tasks.add_task(_run_generation, job_id, req)
    return JobStatus(**jobs[job_id])


print("FastAPI app defined. Model loaded:", "pipe" in globals() and pipe is not None)

In [ ]:
# ── Cell 6: Start server + Cloudflare Tunnel ───────────────────────────────
import uvicorn, threading, subprocess, time, re

def _run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

t = threading.Thread(target=_run_uvicorn, daemon=True)
t.start()
time.sleep(2)

proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

public_url = None
print("Waiting for tunnel URL...")
for line in proc.stdout:
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print()
print("=" * 62)
print(f"  PUBLIC URL:  {public_url}")
print("=" * 62)
print()
print("Paste this URL to Claude Code — it will update the app automatically.")
print(f"\nHealth check: {public_url}/health")

In [ ]:
# ── Cell 7: Keep-alive heartbeat (run IMMEDIATELY after Cell 6) ────────────
# IMPORTANT: run this right after Cell 6 to prevent idle timeout (90 min limit)
import psutil
while True:
    active = sum(1 for j in jobs.values() if j['status'] == 'processing')
    ram = psutil.virtual_memory()
    print(f"[{datetime.utcnow().strftime('%H:%M')}] active={active}  total={len(jobs)}  "
          f"RAM free={ram.available/1e9:.1f}GB  {public_url}")
    time.sleep(270)  # every 4.5 min — under Colab's idle threshold